# Experimento de clasificación CIFAR-10 con CNN

**Proyecto:** Evaluación de Inteligencia Artificial  
**Semilla:** 49  
**Framework:** PyTorch

Este notebook documenta la carga de datos, arquitectura, entrenamiento y evaluación de una red neuronal convolucional.


## 1. Configuración del entorno

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.dataset import SEED, set_seed, load_datasets
from src.model import CNN

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Proyecto:", ROOT)
print("Semilla:", SEED)
print("Dispositivo:", DEVICE)
print("PyTorch:", torch.__version__)


## 2. Carga y distribución de los datos

In [ ]:
(
    train_dataset,
    val_dataset,
    test_dataset,
    train_loader,
    val_loader,
    test_loader,
    base_dataset,
) = load_datasets(root=str(ROOT / "data"), batch_size=128)

print("Entrenamiento:", len(train_dataset))
print("Validación:", len(val_dataset))
print("Prueba:", len(test_dataset))

images, labels = next(iter(train_loader))
print("Forma de imágenes:", tuple(images.shape))
print("Forma de etiquetas:", tuple(labels.shape))


## 3. Arquitectura de la CNN

In [ ]:
model = CNN().to(DEVICE)
parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model)
print(f"Parámetros entrenables: {parameters:,}")

with torch.no_grad():
    output = model(images[:8].to(DEVICE))

print("Forma de salida:", tuple(output.shape))


## 4. Historial de entrenamiento

El historial se genera mediante `src/train.py`.


In [ ]:
history_path = ROOT / "outputs" / "training_history.csv"

if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["epoch"], history["train_loss"], marker="o", label="Entrenamiento")
    axes[0].plot(history["epoch"], history["val_loss"], marker="o", label="Validación")
    axes[0].set_title("Pérdida por época")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["epoch"], history["train_accuracy"], marker="o", label="Entrenamiento")
    axes[1].plot(history["epoch"], history["val_accuracy"], marker="o", label="Validación")
    axes[1].set_title("Exactitud por época")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("No existe training_history.csv. Ejecuta python -m src.train")


## 5. Resultados de prueba

In [ ]:
metrics_path = ROOT / "outputs" / "test_metrics.txt"

if metrics_path.exists():
    print(metrics_path.read_text(encoding="utf-8"))
else:
    print("No existen métricas. Ejecuta python -m src.evaluate")


## 6. Matriz de confusión

In [ ]:
matrix_path = ROOT / "outputs" / "confusion_matrix.csv"

if matrix_path.exists():
    matrix = pd.read_csv(matrix_path, index_col=0)
    display(matrix)

    plt.figure(figsize=(9, 7))
    plt.imshow(matrix.values, cmap="Blues")
    plt.title("Matriz de confusión CIFAR-10")
    plt.xlabel("Clase predicha")
    plt.ylabel("Clase real")
    plt.xticks(range(len(matrix.columns)), matrix.columns, rotation=45, ha="right")
    plt.yticks(range(len(matrix.index)), matrix.index)
    plt.colorbar(label="Cantidad")
    plt.tight_layout()
    plt.show()
else:
    print("No existe confusion_matrix.csv. Ejecuta python -m src.evaluate")


## 7. Conclusiones

- Registrar la exactitud final obtenida en el conjunto de prueba.
- Comparar las curvas de entrenamiento y validación para estudiar la generalización.
- Identificar las clases con menor exactitud y los errores más frecuentes.
- Proponer mejoras futuras, como ajustar las épocas, la arquitectura o la regularización.
